In [ ]:
import joblib  # This is the tool that saves classical ML models
import os
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier


BASE_DIR = "Dataset"
IMG_SIZE = 64  
DATASET_SIZE = 5000
FILTERS = ["Raw", "Gaussian", "Median", "Bilateral", "Combined"]

results_log = []


def apply_filter(img, filter_type):
    if filter_type == "Raw": return img
    elif filter_type == "Gaussian": return cv2.GaussianBlur(img, (5, 5), 0)
    elif filter_type == "Median": return cv2.medianBlur(img, 5)
    elif filter_type == "Bilateral": return cv2.bilateralFilter(img, 9, 75, 75)
    elif filter_type == "Combined":
        img = cv2.GaussianBlur(img, (5, 5), 0)
        img = cv2.medianBlur(img, 5)
        return cv2.bilateralFilter(img, 9, 75, 75)
    return img

def load_data(total_needed, filter_name):
    print(f"Loading {total_needed} images ({filter_name})...")
    data, labels = [], []
    per_class = total_needed // 2
    categories = ["no", "yes"] 
    
    for label_code, category in enumerate(categories):
        folder_path = os.path.join(BASE_DIR, category)
        if not os.path.exists(folder_path): 
            print(f"Error: Folder '{folder_path}' missing.")
            return np.array([]), np.array([])
            
        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:per_class]
        
        for file in files:
            img = cv2.imread(os.path.join(folder_path, file), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = apply_filter(img, filter_name)
                data.append(img.flatten())
                labels.append(label_code)
                
    return np.array(data), np.array(labels)

def build_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
        layers.MaxPooling2D((2, 2)),
        
        
        layers.Conv2D(64, (3, 3), activation='relu', name='last_conv_layer'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5), 
        layers.Dense(1, activation='sigmoid') 
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


print("\n--- PHASE 1: RANDOM FOREST (TESTING ALL FILTERS) ---")

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

for filter_type in FILTERS:
    X, y = load_data(DATASET_SIZE, filter_type)
    if len(X) == 0: continue
        
    # Strict 70/15/15 Split
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(15/85), random_state=42, stratify=y_temp)
    
    # Train and Predict 
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    
    results_log.append({
        "Model": "Random Forest",
        "Filter Type": filter_type,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0)
    })


print("\n--- PHASE 2: DEEP LEARNING (CNN) ---")

# Deep Learning strictly uses Raw data
X_flat, y = load_data(DATASET_SIZE, "Raw")

if len(X_flat) > 0:
    # Unflatten and Normalize (Scale 0-255 down to 0.0-1.0)
    X_cnn = X_flat.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
    
    # Strict 70/15/15 Split
    X_temp, X_test, y_temp, y_test = train_test_split(X_cnn, y, test_size=0.15, random_state=42, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(15/85), random_state=42, stratify=y_temp)
    
    cnn_model = build_cnn()
    
    # Proper training with 20 epochs and Validation monitoring
    print("Training CNN on Raw data... (This will take a minute)")
    cnn_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, batch_size=32, verbose=1)
    
    y_pred_probs = cnn_model.predict(X_test, verbose=0)
    y_pred = np.round(y_pred_probs).flatten()
    
    results_log.append({
        "Model": "CNN",
        "Filter Type": "Raw (Normalized)",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0)
    })
    
    # Clear memory
    tf.keras.backend.clear_session()


print("\n=== FINAL KoS3 EXPERIMENTAL RESULTS ===")
if len(results_log) == 0:
    print("CRITICAL ERROR: No models were trained! Check your dataset folders.")
else:
    df_results = pd.DataFrame(results_log)
    
    # Reorder columns for clean reading
    df_results = df_results[["Model", "Filter Type", "Accuracy", "Precision", "Recall", "F1 Score"]]
    
    # Sort by best F1 Score at the top
    df_results = df_results.sort_values(by="F1 Score", ascending=False).reset_index(drop=True)
    display(df_results)


print("\nSaving trained models to your hard drive...")

# Save the Random Forest brain
joblib.dump(rf_model, 'kos3_random_forest.pkl')

# Save the CNN brain
cnn_model.save('kos3_cnn.keras')

print("✅ Models saved successfully! You are ready to run the GUI.")    


--- PHASE 1: RANDOM FOREST (TESTING ALL FILTERS) ---
Loading 5000 images (Raw)...
Loading 5000 images (Gaussian)...
Loading 5000 images (Median)...
Loading 5000 images (Bilateral)...
Loading 5000 images (Combined)...

--- PHASE 2: DEEP LEARNING (CNN) ---
Loading 5000 images (Raw)...


c:\Users\Tokunbo\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training CNN on Raw data... (This will take a minute)
Epoch 1/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 10s 47ms/step - accuracy: 0.8169 - loss: 0.3997 - val_accuracy: 0.8933 - val_loss: 0.2752
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.9306 - loss: 0.2217 - val_accuracy: 0.9560 - val_loss: 0.1704
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 54ms/step - accuracy: 0.9331 - loss: 0.1865 - val_accuracy: 0.9547 - val_loss: 0.1475
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 56ms/step - accuracy: 0.9483 - loss: 0.1572 - val_accuracy: 0.9467 - val_loss: 0.1595
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - accuracy: 0.9546 - loss: 0.1363 - val_accuracy: 0.9600 - val_loss: 0.1309
Epoch 6/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.9603 - loss: 0.1208 - val_accuracy: 0.9640 - val_loss: 0.1257
Epoch 7/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.9609 - loss: 0.1064 - val_accuracy: 0.9627 - val_loss: 0.1368
Epoch 8/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 5s

,Model,Filter Type,Accuracy,Precision,Recall,F1 Score
0,CNN,Raw (Normalized),0.986667,0.986667,0.986667,0.986667
1,Random Forest,Combined,0.978667,0.978667,0.978667,0.978667
2,Random Forest,Gaussian,0.978667,0.981233,0.976000,0.978610
3,Random Forest,Median,0.977333,0.978610,0.976000,0.977303
4,Random Forest,Raw,0.976000,0.981132,0.970667,0.975871
5,Random Forest,Bilateral,0.976000,0.981132,0.970667,0.975871



Saving trained models to your hard drive...
✅ Models saved successfully! You are ready to run the GUI.
